# Can a finite machine check your brackets?

A finite automaton can count only up to the number of states it has. A stack
counts without bound.

In [1]:
def balanced_with_stack(s):
    stack = []
    for ch in s:
        if ch == "(":
            stack.append(ch)
        elif ch == ")":
            if not stack:
                return False           # pop on an empty stack
            stack.pop()
    return not stack                   # everything closed

for s in ["(()(()))", "(()", "())(", "", "((((((((()))))))))"]:
    print(f"{s!r:22} {balanced_with_stack(s)}")

'(()(()))'             True
'(()'                  False
'())('                 False
''                     True
'((((((((()))))))))'   True


## The best a finite automaton can do

With $k$ states it can track the depth up to $k$. Here is a "counter automaton"
with 3 states, and the input that breaks it.

In [2]:
def balanced_with_k_states(s, k):
    depth = 0
    for ch in s:
        if ch == "(":
            depth = min(depth + 1, k)   # a finite machine cannot remember more than k
        elif ch == ")":
            if depth == 0:
                return False
            depth -= 1
    return depth == 0

k = 3
for s in ["(())", "((()))", "(((())))", "((((()))"]:
    print(f"k = {k}: {s!r:12} stack says {balanced_with_stack(s)!s:5}  {k}-state machine says {balanced_with_k_states(s, k)}")

k = 3: '(())'       stack says True   3-state machine says True
k = 3: '((()))'     stack says True   3-state machine says True
k = 3: '(((())))'   stack says True   3-state machine says False
k = 3: '((((()))'   stack says False  3-state machine says True


Whatever $k$ you choose, the string with $k+1$ opening brackets fools it: it
cannot tell depth $k$ from depth $k+1$. Formally, $a^n b^n$ is not a regular
language.

## The grammar that generates the language

In [3]:
import random
rnd = random.Random(4)

def generate(depth=0):
    """S -> ( S ) S | empty"""
    if depth > 3 or (depth > 0 and rnd.random() < 0.3):
        return ""
    return "(" + generate(depth + 1) + ")" + generate(depth + 1)

samples = [generate() for _ in range(6)]
print(samples)
print(all(balanced_with_stack(s) for s in samples))

['()', '(())((())())', '(())', '()((())())', '(((())())()())(()())', '(((())())(())())((())())()()']
True


**What we used:** the limits of finite automata, push-down automata and
context-free grammars (session 9, sections 2, 4 and 5).